In [1]:
# ============================================================
# BLOCK 1: IMPORTS
# ============================================================

import os
import json
import random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from faker import Faker
from tqdm import tqdm

In [2]:
# ============================================================
# BLOCK 2: CONFIGURATION
# ============================================================

CONFIG = {
    # Root folder that contains images_001 ... images_012
    "dataset_root": r"archive",

    # NIH split files
    "train_val_list": r"train_val_list.txt",
    "test_list": r"test_list.txt",

    # Output folder
    "output_root": r"prismnet_synthetic_output",

    # Fraction of train_val that becomes validation
    "val_ratio_from_trainval": 0.15,

    # Small debug subset
    "small_debug_source_count": 100,

    # Number of synthetic variants per source image
    "variants_per_source_train": 6,
    "variants_per_source_val": 2,
    "variants_per_source_test": 2,
    "variants_per_source_small_debug": 2,

    # Number of PHI and non-PHI overlays per synthetic image
    "min_phi_instances": 2,
    "max_phi_instances": 6,
    "min_nonphi_instances": 1,
    "max_nonphi_instances": 3,

    # Privacy budgets
    "budgets": [0.25, 0.50, 0.75, 1.00],

    # Hospital policies
    "policies": ["hospital_A", "hospital_B", "hospital_C"],

    # Reproducibility
    "seed": 42,

    # Supported image extensions
    "image_exts": [".png", ".jpg", ".jpeg"],

    # Resize option, keep None to preserve original image size
    "resize_to": None,

    # Font size relative to image height
    "font_size_min_frac": 0.018,
    "font_size_max_frac": 0.035,

    # Placement tries
    "max_placement_tries": 50,

    # Save category masks
    "save_category_masks": True,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

In [3]:
# ============================================================
# BLOCK 3: FAKER SETUP
# ============================================================

FAKER_LOCALES = ["en_US", "en_GB", "en_CA", "en_IN"]
FAKERS = [Faker(loc) for loc in FAKER_LOCALES]

for fk in FAKERS:
    fk.seed_instance(CONFIG["seed"])

def choose_faker():
    return random.choice(FAKERS)

In [4]:
# ============================================================
# BLOCK 4: PHI CATEGORIES, BUDGETS, AND POLICIES
# ============================================================

PHI_CATEGORIES = [
    "patient_name",
    "patient_id",
    "dob",
    "exam_date",
    "exam_time",
    "hospital_name",
    "physician_name",
    "age_sex",
]

BUDGET_CATEGORY_MAP = {
    0.25: ["patient_name", "patient_id"],
    0.50: ["patient_name", "patient_id", "dob"],
    0.75: ["patient_name", "patient_id", "dob", "exam_date", "physician_name"],
    1.00: PHI_CATEGORIES[:],
}

POLICY_CATEGORY_MAP = {
    "hospital_A": ["patient_name", "patient_id", "dob"],
    "hospital_B": ["patient_name", "patient_id", "exam_date", "hospital_name"],
    "hospital_C": PHI_CATEGORIES[:],
}

In [5]:
# ============================================================
# BLOCK 5: DIVERSE TEXT COMPONENTS
# ============================================================

HOSPITAL_PREFIXES = [
    "Mercy", "Saint Mary", "Saint Luke", "County", "Regional", "Metro",
    "Northside", "Southside", "Westlake", "Eastview", "Valley", "Central",
    "University", "Community", "Grand", "River", "Lake", "Pine", "Oak",
    "Summit", "Hope", "Providence", "Heritage", "Unity", "General",
    "Midwest", "Lakeside", "Greenview", "Hillcrest", "Brookside",
    "Silverline", "Royal", "National", "Premier", "Harbor", "Crescent"
]

HOSPITAL_MIDDLES = [
    "Medical", "Diagnostic", "Radiology", "Pulmonary", "Chest", "Imaging",
    "Clinical", "Health", "Emergency", "Specialty", "Memorial",
    "Cardio", "Oncology", "Trauma", "Care", "Thoracic"
]

HOSPITAL_SUFFIXES = [
    "Center", "Hospital", "Clinic", "Institute", "Facility",
    "Medical Center", "Diagnostic Center", "Imaging Center",
    "Research Center", "Chest Institute", "Health System"
]

DEPARTMENTS = [
    "Radiology", "Pulmonology", "Emergency", "ICU", "Internal Medicine",
    "Chest Clinic", "Imaging Center", "Thoracic Unit", "Outpatient Imaging",
    "Critical Care", "Respiratory Care"
]

NON_PHI_MARKERS = [
    "AP", "PA", "LAT", "PORTABLE", "UPRIGHT", "SUPINE",
    "CHEST", "L", "R", "LEFT", "RIGHT"
]

In [6]:
# ============================================================
# BLOCK 6: PHI GENERATION FUNCTIONS
# ============================================================

def generate_hospital_name():
    pattern = random.choice([
        "{p} {m} {s}",
        "{p} {s}",
        "{p} {m} Center",
        "{p} {m} Institute",
        "{p} {m} Hospital",
        "{p} {m} and {m2} Center"
    ])
    name = pattern.format(
        p=random.choice(HOSPITAL_PREFIXES),
        m=random.choice(HOSPITAL_MIDDLES),
        m2=random.choice(HOSPITAL_MIDDLES),
        s=random.choice(HOSPITAL_SUFFIXES)
    )
    return " ".join(name.split())

def generate_patient_name(fake: Faker):
    first = fake.first_name()
    last = fake.last_name()
    middle = fake.first_name()

    patterns = [
        lambda: fake.name(),
        lambda: f"{last}, {first}",
        lambda: f"{first} {last}",
        lambda: f"{first} {middle} {last}",
        lambda: f"{first[0]}. {last}",
        lambda: f"{first} {middle[0]}. {last}",
    ]

    name = random.choice(patterns)()
    if random.random() < 0.30:
        name = name.upper()
    return name

def generate_patient_id():
    patterns = [
        lambda: f"MRN:{random.randint(100000, 999999)}",
        lambda: f"PAT ID:{random.randint(1000000, 9999999)}",
        lambda: f"ID {random.randint(10000, 99999)}-{random.randint(10, 99)}",
        lambda: f"ACC:{random.randint(1000, 9999)}-{random.randint(10000, 99999)}",
        lambda: f"XR-{random.randint(1000,9999)}-{random.randint(10,99)}",
        lambda: f"EXAM#{random.randint(100000,999999)}",
    ]
    return random.choice(patterns)()

def generate_date_string(fake: Faker):
    dt = fake.date_between(start_date="-10y", end_date="today")
    formats = [
        "%m/%d/%Y",
        "%Y-%m-%d",
        "%d-%b-%Y",
        "%b %d %Y",
        "%d/%m/%Y",
        "%m-%d-%Y"
    ]
    return pd.Timestamp(dt).strftime(random.choice(formats))

def generate_time_string():
    hh = random.randint(0, 23)
    mm = random.randint(0, 59)
    ss = random.randint(0, 59)
    patterns = [
        f"{hh:02d}:{mm:02d}",
        f"{hh:02d}{mm:02d}",
        f"{hh:02d}:{mm:02d}:{ss:02d}",
        f"{hh:02d}.{mm:02d}",
    ]
    return random.choice(patterns)

def generate_physician_name(fake: Faker):
    last = fake.last_name()
    patterns = [
        lambda: f"Dr. {last}",
        lambda: f"Ref MD: {last}",
        lambda: f"Physician: Dr. {last}",
        lambda: f"Attending: Dr. {last}",
        lambda: f"Reported by Dr. {last}",
    ]
    return random.choice(patterns)()

def generate_age_sex():
    age = random.randint(1, 95)
    sex = random.choice(["M", "F"])
    patterns = [
        f"{age}Y {sex}",
        f"Age:{age} Sex:{sex}",
        f"{sex}/{age}",
        f"{age} {sex}",
    ]
    return random.choice(patterns)

def generate_nonphi_text():
    if random.random() < 0.7:
        return random.choice(NON_PHI_MARKERS)
    return random.choice([
        "CHEST",
        "PORTABLE CHEST",
        "PA CHEST",
        "AP UPRIGHT",
        "INSPIRATION",
        random.choice(DEPARTMENTS),
        f"Dept: {random.choice(DEPARTMENTS)}",
    ])

def generate_phi_pool():
    fake = choose_faker()
    return {
        "patient_name": generate_patient_name(fake),
        "patient_id": generate_patient_id(),
        "dob": f"DOB: {generate_date_string(fake)}",
        "exam_date": f"DATE: {generate_date_string(fake)}",
        "exam_time": f"TIME: {generate_time_string()}",
        "hospital_name": generate_hospital_name(),
        "physician_name": generate_physician_name(fake),
        "age_sex": generate_age_sex(),
    }

In [7]:
# ============================================================
# BLOCK 7: FONT DISCOVERY
# ============================================================

def discover_fonts() -> List[str]:
    possible_dirs = [
        r"C:\Windows\Fonts",
        "/usr/share/fonts",
        "/usr/local/share/fonts",
        str(Path.home() / ".fonts"),
    ]
    font_files = []
    for d in possible_dirs:
        d = Path(d)
        if d.exists():
            for ext in ["*.ttf", "*.otf", "*.ttc"]:
                font_files.extend([str(p) for p in d.rglob(ext)])

    good_keywords = ["arial", "helvetica", "cour", "dejavu", "liberation", "noto", "sans"]
    selected = []
    for f in font_files:
        lf = f.lower()
        if any(k in lf for k in good_keywords):
            selected.append(f)

    return selected[:200] if selected else []

FONT_FILES = discover_fonts()

def load_font(font_size: int):
    if FONT_FILES:
        fpath = random.choice(FONT_FILES)
        try:
            return ImageFont.truetype(fpath, font_size)
        except Exception:
            pass
    return ImageFont.load_default()

In [8]:
# ============================================================
# BLOCK 8: IMAGE DISCOVERY AND SPLIT BUILDING
# ============================================================

def discover_all_images(dataset_root: str, exts: List[str]) -> List[Path]:
    root = Path(dataset_root)
    all_files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            if "images" in [part.lower() for part in p.parts]:
                all_files.append(p)
    return sorted(all_files)

def read_split_list(file_path: str) -> List[str]:
    p = Path(file_path)
    if not p.exists():
        return []
    with open(p, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def build_splits(all_image_paths: List[Path]) -> Dict[str, List[Path]]:
    image_name_to_path = {p.name: p for p in all_image_paths}

    train_val_names = read_split_list(CONFIG["train_val_list"])
    test_names = read_split_list(CONFIG["test_list"])

    if train_val_names and test_names:
        train_val_paths = [image_name_to_path[n] for n in train_val_names if n in image_name_to_path]
        test_paths = [image_name_to_path[n] for n in test_names if n in image_name_to_path]

        random.shuffle(train_val_paths)
        val_size = int(len(train_val_paths) * CONFIG["val_ratio_from_trainval"])
        val_paths = train_val_paths[:val_size]
        train_paths = train_val_paths[val_size:]
    else:
        paths = all_image_paths[:]
        random.shuffle(paths)
        n = len(paths)
        test_size = int(n * 0.20)
        val_size = int(n * 0.10)
        test_paths = paths[:test_size]
        val_paths = paths[test_size:test_size + val_size]
        train_paths = paths[test_size + val_size:]

    small_debug_paths = train_paths[:min(CONFIG["small_debug_source_count"], len(train_paths))]

    return {
        "train": train_paths,
        "val": val_paths,
        "test": test_paths,
        "small_debug_train": small_debug_paths,
    }

In [9]:
# ============================================================
# BLOCK 9: PLACEMENT STRATEGY
# ============================================================

def get_candidate_zones(w: int, h: int) -> Dict[str, Tuple[int, int, int, int]]:
    return {
        "top_left": (0, 0, int(0.28*w), int(0.18*h)),
        "top_right": (int(0.72*w), 0, w, int(0.18*h)),
        "bottom_left": (0, int(0.82*h), int(0.32*w), h),
        "bottom_right": (int(0.68*w), int(0.82*h), w, h),
        "left_margin": (0, int(0.20*h), int(0.18*w), int(0.80*h)),
        "right_margin": (int(0.82*w), int(0.20*h), w, int(0.80*h)),
        "top_center": (int(0.35*w), 0, int(0.65*w), int(0.12*h)),
    }

def boxes_intersect(a, b, pad=4):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 + pad < bx1 or bx2 + pad < ax1 or ay2 + pad < by1 or by2 + pad < ay1)

def sample_position_for_text(zone_box, text_box_size, occupied_boxes, max_tries=50):
    zx1, zy1, zx2, zy2 = zone_box
    tw, th = text_box_size
    if (zx2 - zx1) <= tw or (zy2 - zy1) <= th:
        return None

    for _ in range(max_tries):
        x = random.randint(zx1, max(zx1, zx2 - tw))
        y = random.randint(zy1, max(zy1, zy2 - th))
        candidate = (x, y, x + tw, y + th)
        if not any(boxes_intersect(candidate, occ) for occ in occupied_boxes):
            return candidate
    return None

In [10]:
# ============================================================
# BLOCK 10: TEXT STYLE
# ============================================================

def random_text_style(h: int):
    font_size = random.randint(
        int(CONFIG["font_size_min_frac"] * h),
        int(CONFIG["font_size_max_frac"] * h)
    )
    intensity = random.randint(175, 255)
    opacity = random.uniform(0.70, 1.0)
    angle = random.uniform(-4, 4) if random.random() < 0.35 else 0.0

    return {
        "font_size": font_size,
        "intensity": intensity,
        "opacity": opacity,
        "angle": angle,
    }

In [11]:
# ============================================================
# BLOCK 11: TEXT PATCH RENDERING
# ============================================================

def render_text_patch(text: str, style: Dict):
    font = load_font(style["font_size"])

    tmp_img = Image.new("L", (2000, 400), 0)
    tmp_draw = ImageDraw.Draw(tmp_img)
    bbox = tmp_draw.textbbox((0, 0), text, font=font)

    tw = max(1, bbox[2] - bbox[0] + 4)
    th = max(1, bbox[3] - bbox[1] + 4)

    patch = Image.new("L", (tw, th), 0)
    patch_draw = ImageDraw.Draw(patch)
    patch_draw.text((2, 2), text, fill=int(255 * style["opacity"]), font=font)

    if style["angle"] != 0:
        patch = patch.rotate(style["angle"], expand=True, fillcolor=0)

    return patch

In [12]:
# ============================================================
# BLOCK 12: PASTE TEXT AND UPDATE MASKS
# ============================================================

def paste_text_with_mask(
    base_img: Image.Image,
    text_patch: Image.Image,
    x: int,
    y: int,
    intensity: int,
    global_mask: np.ndarray,
    category_mask: np.ndarray,
):
    patch_np = np.array(text_patch).astype(np.float32) / 255.0
    if patch_np.max() <= 0:
        return

    base_np = np.array(base_img).astype(np.float32)

    ph, pw = patch_np.shape
    H, W = base_np.shape

    x2 = min(W, x + pw)
    y2 = min(H, y + ph)
    x1 = max(0, x)
    y1 = max(0, y)

    px1 = max(0, -x)
    py1 = max(0, -y)
    px2 = px1 + (x2 - x1)
    py2 = py1 + (y2 - y1)

    if x2 <= x1 or y2 <= y1:
        return

    alpha = patch_np[py1:py2, px1:px2]
    region = base_np[y1:y2, x1:x2]

    text_val = np.full_like(region, fill_value=intensity, dtype=np.float32)
    out = region * (1 - alpha) + text_val * alpha
    base_np[y1:y2, x1:x2] = np.clip(out, 0, 255)

    bin_alpha = (alpha > 0.05).astype(np.uint8)
    global_mask[y1:y2, x1:x2] = np.maximum(global_mask[y1:y2, x1:x2], bin_alpha)
    category_mask[y1:y2, x1:x2] = np.maximum(category_mask[y1:y2, x1:x2], bin_alpha)

    base_img.paste(Image.fromarray(base_np.astype(np.uint8)))

In [13]:
# ============================================================
# BLOCK 13: GLOBAL AUGMENTATIONS
# ============================================================

def apply_global_augmentations(img: Image.Image) -> Image.Image:
    if random.random() < 0.35:
        radius = random.uniform(0.3, 1.2)
        img = img.filter(ImageFilter.GaussianBlur(radius=radius))

    if random.random() < 0.35:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.92, 1.08))

    if random.random() < 0.25:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.96, 1.04))

    if random.random() < 0.25:
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(1.0, 4.0), size=arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)

    return img

In [14]:
# ============================================================
# BLOCK 14: TARGET MASK BUILDERS
# ============================================================

def build_budget_mask(category_masks: Dict[str, np.ndarray], budget: float) -> np.ndarray:
    cats = BUDGET_CATEGORY_MAP[budget]
    sample_mask = next(iter(category_masks.values()))
    out = np.zeros_like(sample_mask, dtype=np.uint8)
    for c in cats:
        if c in category_masks:
            out = np.maximum(out, category_masks[c])
    return out

def build_policy_mask(category_masks: Dict[str, np.ndarray], policy_name: str) -> np.ndarray:
    cats = POLICY_CATEGORY_MAP[policy_name]
    sample_mask = next(iter(category_masks.values()))
    out = np.zeros_like(sample_mask, dtype=np.uint8)
    for c in cats:
        if c in category_masks:
            out = np.maximum(out, category_masks[c])
    return out

def build_budget_policy_mask(
    category_masks: Dict[str, np.ndarray],
    budget: float,
    policy_name: str
) -> np.ndarray:
    """
    Final target mask under BOTH:
      - privacy budget
      - hospital policy
    """
    budget_cats = set(BUDGET_CATEGORY_MAP[budget])
    policy_cats = set(POLICY_CATEGORY_MAP[policy_name])
    active_cats = budget_cats.intersection(policy_cats)

    sample_mask = next(iter(category_masks.values()))
    out = np.zeros_like(sample_mask, dtype=np.uint8)

    for cat in active_cats:
        if cat in category_masks:
            out = np.maximum(out, category_masks[cat])

    return out

In [15]:
# ============================================================
# BLOCK 15: GENERATE ONE SYNTHETIC SAMPLE
# ============================================================

def generate_one_sample(
    source_path: Path,
    out_split_dir: Path,
    sample_id: str,
    split_name: str,
):
    img = Image.open(source_path).convert("L")

    if CONFIG["resize_to"] is not None:
        img = img.resize(CONFIG["resize_to"], Image.BILINEAR)

    w, h = img.size
    zones = get_candidate_zones(w, h)

    syn_img = img.copy()

    # Pixel-level masks
    all_phi_mask = np.zeros((h, w), dtype=np.uint8)
    non_phi_mask = np.zeros((h, w), dtype=np.uint8)
    category_masks = {cat: np.zeros((h, w), dtype=np.uint8) for cat in PHI_CATEGORIES}

    # Instance-level records
    instances = []
    occupied_boxes = []

    phi_pool = generate_phi_pool()

    n_phi = random.randint(CONFIG["min_phi_instances"], CONFIG["max_phi_instances"])
    n_nonphi = random.randint(CONFIG["min_nonphi_instances"], CONFIG["max_nonphi_instances"])

    chosen_phi_categories = random.sample(PHI_CATEGORIES, k=min(n_phi, len(PHI_CATEGORIES)))

    # -----------------------------
    # Draw PHI text instances
    # -----------------------------
    for category in chosen_phi_categories:
        text = phi_pool[category]
        style = random_text_style(h)
        patch = render_text_patch(text, style)
        pw, ph = patch.size

        zone_name = random.choice(list(zones.keys()))
        placement = sample_position_for_text(
            zones[zone_name],
            (pw, ph),
            occupied_boxes,
            max_tries=CONFIG["max_placement_tries"],
        )
        if placement is None:
            continue

        x1, y1, x2, y2 = placement

        paste_text_with_mask(
            base_img=syn_img,
            text_patch=patch,
            x=x1,
            y=y1,
            intensity=style["intensity"],
            global_mask=all_phi_mask,
            category_mask=category_masks[category],
        )

        occupied_boxes.append((x1, y1, x2, y2))

        instances.append({
            "instance_id": len(instances),
            "text": text,
            "category": category,
            "is_phi": True,
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "bbox_xywh": [int(x1), int(y1), int(x2 - x1), int(y2 - y1)],
            "font_size": int(style["font_size"]),
            "intensity": int(style["intensity"]),
            "opacity": float(style["opacity"]),
            "angle": float(style["angle"]),
            "zone": zone_name,
        })

    # -----------------------------
    # Draw non-PHI text instances
    # -----------------------------
    for _ in range(n_nonphi):
        text = generate_nonphi_text()
        style = random_text_style(h)
        style["intensity"] = max(150, style["intensity"] - random.randint(0, 20))

        patch = render_text_patch(text, style)
        pw, ph = patch.size

        zone_name = random.choice(list(zones.keys()))
        placement = sample_position_for_text(
            zones[zone_name],
            (pw, ph),
            occupied_boxes,
            max_tries=CONFIG["max_placement_tries"],
        )
        if placement is None:
            continue

        x1, y1, x2, y2 = placement
        dummy_nonphi_cat = np.zeros((h, w), dtype=np.uint8)

        paste_text_with_mask(
            base_img=syn_img,
            text_patch=patch,
            x=x1,
            y=y1,
            intensity=style["intensity"],
            global_mask=non_phi_mask,
            category_mask=dummy_nonphi_cat,
        )

        occupied_boxes.append((x1, y1, x2, y2))

        instances.append({
            "instance_id": len(instances),
            "text": text,
            "category": "non_phi_marker",
            "is_phi": False,
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "bbox_xywh": [int(x1), int(y1), int(x2 - x1), int(y2 - y1)],
            "font_size": int(style["font_size"]),
            "intensity": int(style["intensity"]),
            "opacity": float(style["opacity"]),
            "angle": float(style["angle"]),
            "zone": zone_name,
        })

    syn_img = apply_global_augmentations(syn_img)

    # -----------------------------
    # Save synthetic image
    # -----------------------------
    image_dir = out_split_dir / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    image_path = image_dir / f"{sample_id}.png"
    syn_img.save(image_path)

    # -----------------------------
    # Save pixel-level masks
    # -----------------------------
    mask_root = out_split_dir / "masks"
    all_phi_dir = mask_root / "all_phi"
    non_phi_dir = mask_root / "non_phi_text"
    all_phi_dir.mkdir(parents=True, exist_ok=True)
    non_phi_dir.mkdir(parents=True, exist_ok=True)

    all_phi_path = all_phi_dir / f"{sample_id}.png"
    non_phi_path = non_phi_dir / f"{sample_id}.png"

    Image.fromarray((all_phi_mask * 255).astype(np.uint8)).save(all_phi_path)
    Image.fromarray((non_phi_mask * 255).astype(np.uint8)).save(non_phi_path)

    category_mask_paths = {}
    if CONFIG["save_category_masks"]:
        for cat, cmask in category_masks.items():
            if cmask.sum() > 0:
                cat_dir = mask_root / "categories" / cat
                cat_dir.mkdir(parents=True, exist_ok=True)
                cat_path = cat_dir / f"{sample_id}.png"
                Image.fromarray((cmask * 255).astype(np.uint8)).save(cat_path)
                category_mask_paths[cat] = str(cat_path)

    # -----------------------------
    # Save budget-conditioned masks
    # -----------------------------
    budget_mask_paths = {}
    budget_root = out_split_dir / "targets" / "budgets"
    for b in CONFIG["budgets"]:
        bmask = build_budget_mask(category_masks, b)
        bdir = budget_root / f"budget_{str(b).replace('.', '_')}"
        bdir.mkdir(parents=True, exist_ok=True)
        bpath = bdir / f"{sample_id}.png"
        Image.fromarray((bmask * 255).astype(np.uint8)).save(bpath)
        budget_mask_paths[str(b)] = str(bpath)

    # -----------------------------
    # Save policy-conditioned masks
    # -----------------------------
    policy_mask_paths = {}
    policy_root = out_split_dir / "targets" / "policies"
    for pol in CONFIG["policies"]:
        pmask = build_policy_mask(category_masks, pol)
        pdir = policy_root / pol
        pdir.mkdir(parents=True, exist_ok=True)
        ppath = pdir / f"{sample_id}.png"
        Image.fromarray((pmask * 255).astype(np.uint8)).save(ppath)
        policy_mask_paths[pol] = str(ppath)

    # -----------------------------
    # Save combined budget + policy masks
    # -----------------------------
    combined_mask_paths = {}
    combined_root = out_split_dir / "targets" / "budget_policy_combined"

    for b in CONFIG["budgets"]:
        bkey = str(b)
        combined_mask_paths[bkey] = {}

        for pol in CONFIG["policies"]:
            cmask = build_budget_policy_mask(category_masks, b, pol)

            combo_dir = combined_root / f"budget_{str(b).replace('.', '_')}" / pol
            combo_dir.mkdir(parents=True, exist_ok=True)

            cpath = combo_dir / f"{sample_id}.png"
            Image.fromarray((cmask * 255).astype(np.uint8)).save(cpath)

            combined_mask_paths[bkey][pol] = str(cpath)

    # -----------------------------
    # Count summary per image
    # -----------------------------
    phi_instance_count = sum(1 for x in instances if x["is_phi"])
    non_phi_instance_count = sum(1 for x in instances if not x["is_phi"])

    # -----------------------------
    # Rich metadata returned
    # -----------------------------
    meta = {
        "sample_id": sample_id,
        "split": split_name,
        "source_image_name": source_path.name,
        "source_image_path": str(source_path),
        "synthetic_image_path": str(image_path),
        "width": w,
        "height": h,
        "phi_instance_count": phi_instance_count,
        "non_phi_instance_count": non_phi_instance_count,
        "instances": instances,
        "mask_paths": {
            "all_phi": str(all_phi_path),
            "non_phi_text": str(non_phi_path),
            "categories": category_mask_paths,
            "budgets": budget_mask_paths,
            "policies": policy_mask_paths,
            "budget_policy_combined": combined_mask_paths,
        },
        "budgets": CONFIG["budgets"],
        "policies": CONFIG["policies"],
    }

    return meta

In [16]:
# ============================================================
# BLOCK 16: GENERATE A FULL SPLIT + SAVE ANNOTATION TABLES
# ============================================================

def generate_split_dataset(split_name: str, source_paths: List[Path], variants_per_source: int, out_root: Path):
    out_split_dir = out_root / split_name
    out_split_dir.mkdir(parents=True, exist_ok=True)

    metadata_file = out_split_dir / "metadata.jsonl"
    flat_csv_file = out_split_dir / "annotations_flat.csv"

    if metadata_file.exists():
        metadata_file.unlink()
    if flat_csv_file.exists():
        flat_csv_file.unlink()

    total_samples = len(source_paths) * variants_per_source
    pbar = tqdm(total=total_samples, desc=f"Generating {split_name}")

    flat_rows = []

    with open(metadata_file, "w", encoding="utf-8") as mf:
        for src_path in source_paths:
            stem = src_path.stem

            for v in range(variants_per_source):
                sample_id = f"{stem}_syn_{v:02d}"

                meta = generate_one_sample(
                    source_path=src_path,
                    out_split_dir=out_split_dir,
                    sample_id=sample_id,
                    split_name=split_name,
                )

                mf.write(json.dumps(meta) + "\n")

                for inst in meta["instances"]:
                    row = {
                        "sample_id": meta["sample_id"],
                        "split": meta["split"],
                        "source_image_name": meta["source_image_name"],
                        "synthetic_image_path": meta["synthetic_image_path"],
                        "instance_id": inst["instance_id"],
                        "text": inst["text"],
                        "category": inst["category"],
                        "is_phi": inst["is_phi"],
                        "x1": inst["bbox"][0],
                        "y1": inst["bbox"][1],
                        "x2": inst["bbox"][2],
                        "y2": inst["bbox"][3],
                        "x": inst["bbox_xywh"][0],
                        "y": inst["bbox_xywh"][1],
                        "w": inst["bbox_xywh"][2],
                        "h": inst["bbox_xywh"][3],
                        "font_size": inst["font_size"],
                        "intensity": inst["intensity"],
                        "opacity": inst["opacity"],
                        "angle": inst["angle"],
                        "zone": inst["zone"],
                        "all_phi_mask_path": meta["mask_paths"]["all_phi"],
                        "non_phi_mask_path": meta["mask_paths"]["non_phi_text"],
                    }
                    flat_rows.append(row)

                pbar.update(1)

    pbar.close()

    if flat_rows:
        df = pd.DataFrame(flat_rows)
        df.to_csv(flat_csv_file, index=False)

In [17]:
# ============================================================
# BLOCK 17: MAIN FUNCTION
# ============================================================

def main():
    dataset_root = CONFIG["dataset_root"]
    output_root = Path(CONFIG["output_root"])

    if not output_root.exists():
        output_root.mkdir(parents=True, exist_ok=True)

    all_images = discover_all_images(dataset_root, CONFIG["image_exts"])
    print(f"[INFO] Found {len(all_images)} source images.")

    if len(all_images) == 0:
        raise RuntimeError("No images found. Check dataset_root and folder structure.")

    splits = build_splits(all_images)

    print(f"[INFO] Train sources: {len(splits['train'])}")
    print(f"[INFO] Val sources: {len(splits['val'])}")
    print(f"[INFO] Test sources: {len(splits['test'])}")
    print(f"[INFO] Small debug train sources: {len(splits['small_debug_train'])}")

    manifest_dir = output_root / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)

    for split_name, paths in splits.items():
        with open(manifest_dir / f"{split_name}_sources.txt", "w", encoding="utf-8") as f:
            for p in paths:
                f.write(str(p) + "\n")

    generate_split_dataset(
        split_name="train",
        source_paths=splits["train"],
        variants_per_source=CONFIG["variants_per_source_train"],
        out_root=output_root,
    )

    generate_split_dataset(
        split_name="val",
        source_paths=splits["val"],
        variants_per_source=CONFIG["variants_per_source_val"],
        out_root=output_root,
    )

    generate_split_dataset(
        split_name="test",
        source_paths=splits["test"],
        variants_per_source=CONFIG["variants_per_source_test"],
        out_root=output_root,
    )

    generate_split_dataset(
        split_name="small_debug_train",
        source_paths=splits["small_debug_train"],
        variants_per_source=CONFIG["variants_per_source_small_debug"],
        out_root=output_root,
    )

    with open(output_root / "generation_config.json", "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=2)

    print("\n[DONE] Synthetic data generation completed.")
    print(f"[DONE] Output saved to: {output_root.resolve()}")

In [ ]:
# ============================================================
# BLOCK 18: ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()

[INFO] Found 112120 source images.
[INFO] Train sources: 78484
[INFO] Val sources: 11212
[INFO] Test sources: 22424
[INFO] Small debug train sources: 100


Generating train:   0%|          | 2/470904 [00:02<140:13:05,  1.07s/it]